In [79]:
import pandas as pd
import numpy as np
from ast import literal_eval

from tqdm import tqdm
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")


In [80]:
users = pd.read_csv("/content/users.tsv", sep="\t")
history = pd.read_csv("/content/history.tsv", sep="\t")
validate = pd.read_csv("/content/validate.tsv", sep="\t")
answers = pd.read_csv("/content/validate_answers.tsv", sep="\t")


In [81]:
users['age_group'] = users['age'].replace(0, -1)
users['sex'] = users['sex'].astype('category')
users['city_id'] = users['city_id'].astype('category')


In [82]:
users

,user_id,sex,age,city_id,age_group
0,0,2,19,0,19
1,1,1,0,1,-1
2,2,2,24,2,24
3,3,1,20,3,20
4,4,2,29,4,29
...,...,...,...,...,...
27764,27764,1,38,295,38
27765,27765,2,30,79,30
27766,27766,2,21,1953,21
27767,27767,2,17,0,17


In [83]:
user_agg = history.groupby("user_id").agg(total_impressions = ("hour", "count"), mean_cpm = ("cpm", "mean"), median_cpm = ("cpm", "median"), max_cpm = ("cpm", "max")).reset_index()
user_agg["user_id"] = user_agg["user_id"].astype(int)
user_agg

,user_id,total_impressions,mean_cpm,median_cpm,max_cpm
0,1,12,99.850000,66.600,285.00
1,3,1,312.500000,312.500,312.50
2,4,22,34.631818,30.000,90.00
3,5,2,76.000000,76.000,120.00
4,6,8,307.933750,255.800,696.87
...,...,...,...,...,...
18212,27763,3,86.166667,90.000,117.50
18213,27764,4,160.375000,161.750,210.00
18214,27765,6,148.133333,125.000,276.00
18215,27766,12,125.197500,122.315,224.67


In [84]:
all_users = users[["user_id"]].copy()
all_users["user_id"] = all_users["user_id"].astype(int)

user_agg = all_users.merge(user_agg, on="user_id", how="left")
user_agg.fillna(0, inplace=True)
user_agg # для пользователей, у которых нет истории зануляем признаки

,user_id,total_impressions,mean_cpm,median_cpm,max_cpm
0,0,0.0,0.000000,0.000,0.00
1,1,12.0,99.850000,66.600,285.00
2,2,0.0,0.000000,0.000,0.00
3,3,1.0,312.500000,312.500,312.50
4,4,22.0,34.631818,30.000,90.00
...,...,...,...,...,...
27764,27764,4.0,160.375000,161.750,210.00
27765,27765,6.0,148.133333,125.000,276.00
27766,27766,12.0,125.197500,122.315,224.67
27767,27767,0.0,0.000000,0.000,0.00


In [85]:
publisher_agg = history.groupby("publisher").agg(total_impressions = ("user_id", "count"), mean_cpm = ("cpm", "mean")).reset_index()
publisher_agg["publisher"] = publisher_agg["publisher"].astype(int)
publisher_agg

,publisher,total_impressions,mean_cpm
0,1,111042,172.664278
1,2,44724,195.638455
2,3,10700,189.964138
3,4,393,91.746387
4,5,1158,183.727910
5,6,998,84.014890
6,7,10722,228.330933
7,8,548,106.718157
8,9,1141,92.119684
9,10,640,83.430188


в предыдущем ноутбуке я уже посмотрела, что данных в history.tsv было больше миллиона, для дл нам нужны эмбеддинги(короткий набор чиселок), поэтому я сделала агрегат для пользователя(сколько раз видел рекламу, характеристики cpm) и для кампаний(средняя ставка и количество показов)

In [86]:
validate['duration'] = validate['hour_end'] - validate['hour_start']
validate['hour_start_of_day'] = validate['hour_start'] % 24
validate['hour_end_of_day'] = validate['hour_end'] % 24

In [87]:
validate

,cpm,hour_start,hour_end,publishers,audience_size,user_ids,duration,hour_start_of_day,hour_end_of_day
0,220.0,1058,1153,"7,17",1906,"12,44,46,50,58,71,93,122,134,143,176,184,187,1...",95,2,1
1,312.0,1295,1301,"3,18",1380,"29,81,98,102,165,167,195,205,218,231,242,263,3...",6,23,5
2,70.0,1229,1249,"1,2,3,9,15,21",888,"12,23,25,29,45,85,92,124,156,190,272,334,456,5...",20,5,1
3,240.0,1295,1377,"1,14",440,"44,122,187,209,242,255,312,345,382,465,513,524...",82,23,9
4,262.0,752,990,"1,3,7,8",1476,"15,24,30,43,50,53,96,105,159,168,181,190,196,2...",238,8,6
...,...,...,...,...,...,...,...,...,...
516,116.0,878,883,"2,3,7",656,"4,38,179,180,222,346,404,408,430,449,494,635,6...",5,14,19
517,138.0,1395,1442,"1,2,5,14,16",808,"42,44,66,90,221,325,348,354,401,404,428,484,52...",47,3,2
518,143.0,1164,1184,"2,3,7,14,16,17",1500,"0,36,80,85,114,140,148,153,176,189,209,220,233...",20,12,8
519,130.0,883,997,"1,7,21",348,"121,304,402,417,510,545,548,555,568,661,986,10...",114,19,13


In [88]:
validate["user_ids"] = validate["user_ids"].apply(lambda x: [int(i) for i in x.split(",")])
validate["publishers"] = validate["publishers"].apply(lambda x: [int(i) for i in x.split(",")])

теперь из добавленного мы видим, когда в каком часу кампания началась/закончилась, а также кампании и user из них теперь кортежи, а не строки.

In [92]:
def agg_aud_features(user_ids, user_agg):
    aud = user_agg[user_agg["user_id"].isin(user_ids)]

    if len(aud) == 0:
        return pd.Series({
            "aud_mean_imp": 0.0,
            "aud_median_imp": 0.0,
            "aud_mean_cpm": 0.0,
            "aud_median_cpm": 0.0,
            "aud_max_cpm": 0.0
        })

    return pd.Series({
        "aud_mean_imp": aud["total_impressions"].mean(),
        "aud_median_imp": aud["total_impressions"].median(),
        "aud_mean_cpm": aud["mean_cpm"].mean(),
        "aud_median_cpm": aud["median_cpm"].mean(),
        "aud_max_cpm": aud["max_cpm"].max()
    })


In [94]:
audience_features = validate["user_ids"].apply( lambda ids: agg_aud_features(ids, user_agg))
audience_features

,aud_mean_imp,aud_median_imp,aud_mean_cpm,aud_median_cpm,aud_max_cpm
0,6.663694,2.0,177.394734,154.948940,56579.07
1,19.973913,16.0,163.268813,123.862554,36395.60
2,6.890766,2.0,162.120671,137.818840,21077.38
3,6.465909,2.0,161.117249,137.290670,56579.07
4,5.533875,1.0,190.072762,165.578157,38282.14
...,...,...,...,...,...
516,19.661585,16.0,167.358729,128.366052,21077.38
517,6.370050,2.0,146.303113,124.898100,14697.84
518,6.762667,2.0,155.806314,132.065047,38282.14
519,6.589080,2.0,156.621980,138.905043,21077.38


In [96]:
validate = pd.concat([validate, audience_features], axis=1)


In [97]:
validate.isna().sum()


,0
cpm,0
hour_start,0
hour_end,0
publishers,0
audience_size,0
user_ids,0
duration,0
hour_start_of_day,0
hour_end_of_day,0
aud_mean_imp,0


In [98]:
validate.describe()


,cpm,hour_start,hour_end,audience_size,duration,hour_start_of_day,hour_end_of_day,aud_mean_imp,aud_median_imp,aud_mean_cpm,aud_median_cpm,aud_max_cpm,aud_mean_imp,aud_median_imp,aud_mean_cpm,aud_median_cpm,aud_max_cpm
count,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000,521.000000
mean,165.105566,1061.913628,1164.541267,1074.477927,102.627639,11.303263,11.067179,6.686478,2.608445,156.700607,132.553363,27927.059463,6.686478,2.608445,156.700607,132.553363,27927.059463
std,112.425695,190.464408,190.218292,594.415948,124.011908,7.035909,7.015731,4.653815,4.361352,78.340964,67.651116,21939.434431,4.653815,4.361352,78.340964,67.651116,21939.434431
min,30.000000,748.000000,755.000000,300.000000,2.000000,0.000000,0.000000,0.154000,0.000000,26.151191,22.588064,625.070000,0.154000,0.000000,26.151191,22.588064,625.070000
25%,80.000000,908.000000,1022.000000,500.000000,8.000000,5.000000,5.000000,5.417355,1.000000,125.268552,100.131005,6340.250000,5.417355,1.000000,125.268552,100.131005,6340.250000
50%,130.000000,1044.000000,1171.000000,960.000000,41.000000,11.000000,11.000000,6.454545,2.000000,151.223980,126.978260,22720.950000,6.454545,2.000000,151.223980,126.978260,22720.950000
75%,231.000000,1190.000000,1319.000000,1428.000000,177.000000,17.000000,17.000000,7.069969,2.000000,172.135107,142.590076,54373.770000,7.069969,2.000000,172.135107,142.590076,54373.770000
max,475.000000,1485.000000,1488.000000,2500.000000,497.000000,23.000000,23.000000,30.952222,28.000000,504.896075,407.784745,64282.980000,30.952222,28.000000,504.896075,407.784745,64282.980000


In [99]:
validate

,cpm,hour_start,hour_end,publishers,audience_size,user_ids,duration,hour_start_of_day,hour_end_of_day,aud_mean_imp,aud_median_imp,aud_mean_cpm,aud_median_cpm,aud_max_cpm,aud_mean_imp,aud_median_imp,aud_mean_cpm,aud_median_cpm,aud_max_cpm
0,220.0,1058,1153,"[7, 17]",1906,"[12, 44, 46, 50, 58, 71, 93, 122, 134, 143, 17...",95,2,1,6.663694,2.0,177.394734,154.948940,56579.07,6.663694,2.0,177.394734,154.948940,56579.07
1,312.0,1295,1301,"[3, 18]",1380,"[29, 81, 98, 102, 165, 167, 195, 205, 218, 231...",6,23,5,19.973913,16.0,163.268813,123.862554,36395.60,19.973913,16.0,163.268813,123.862554,36395.60
2,70.0,1229,1249,"[1, 2, 3, 9, 15, 21]",888,"[12, 23, 25, 29, 45, 85, 92, 124, 156, 190, 27...",20,5,1,6.890766,2.0,162.120671,137.818840,21077.38,6.890766,2.0,162.120671,137.818840,21077.38
3,240.0,1295,1377,"[1, 14]",440,"[44, 122, 187, 209, 242, 255, 312, 345, 382, 4...",82,23,9,6.465909,2.0,161.117249,137.290670,56579.07,6.465909,2.0,161.117249,137.290670,56579.07
4,262.0,752,990,"[1, 3, 7, 8]",1476,"[15, 24, 30, 43, 50, 53, 96, 105, 159, 168, 18...",238,8,6,5.533875,1.0,190.072762,165.578157,38282.14,5.533875,1.0,190.072762,165.578157,38282.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,116.0,878,883,"[2, 3, 7]",656,"[4, 38, 179, 180, 222, 346, 404, 408, 430, 449...",5,14,19,19.661585,16.0,167.358729,128.366052,21077.38,19.661585,16.0,167.358729,128.366052,21077.38
517,138.0,1395,1442,"[1, 2, 5, 14, 16]",808,"[42, 44, 66, 90, 221, 325, 348, 354, 401, 404,...",47,3,2,6.370050,2.0,146.303113,124.898100,14697.84,6.370050,2.0,146.303113,124.898100,14697.84
518,143.0,1164,1184,"[2, 3, 7, 14, 16, 17]",1500,"[0, 36, 80, 85, 114, 140, 148, 153, 176, 189, ...",20,12,8,6.762667,2.0,155.806314,132.065047,38282.14,6.762667,2.0,155.806314,132.065047,38282.14
519,130.0,883,997,"[1, 7, 21]",348,"[121, 304, 402, 417, 510, 545, 548, 555, 568, ...",114,19,13,6.589080,2.0,156.621980,138.905043,21077.38,6.589080,2.0,156.621980,138.905043,21077.38


In [104]:
validate.to_parquet("/content/validate_features.parquet", index=False)
user_agg.to_parquet("/content/user_aggregates.parquet", index=False)
publisher_agg.to_parquet("/content/publisher_aggregates.parquet", index=False)
